In [ ]:
import socket
import pickle
import struct
import torch
import threading
import torch.nn as nn
import copy
from torch.utils.data import DataLoader
from model import SimpleModel
import time
import argparse
import sys
import traceback
import os
import h5py
from data_utils import read_client_data
from prunning import prune_and_restructure
from size_mode import get_model_size
import builtins
from synexp import LayerComplexityCalculator
import numpy as np
import random
from prunning_nisp import prune_fc1
from prunning_snip import snip_pruning, apply_mask
from thop import profile



In [ ]:
def dequantization(self, global_state):
    dequantized_state_dict = {}
    for k, v in global_state.items():
        if isinstance(v, dict) and v.get('dtype') == 'quantized_int8':
            # Recupera tensores quantizados
            scale = v['scale']
            dequantized_state_dict[k] = v['weights'].float() * scale
        else:
            # Mantém tensores normais
            dequantized_state_dict[k] = v
    return dequantized_state_dict

def quantization(self, state_dict):
    quantized_state_dict = {}
    keys = list(state_dict.keys())
    for k, v in state_dict.items():
        if isinstance(v, torch.Tensor):
            scale = torch.max(torch.abs(v)) / 127.0
            quantized_weights = torch.clamp((v / scale).round(), -128, 127).to(torch.int8)
            quantized_state_dict[k] = {
                'dtype': 'quantized_int8',
                'scale': scale,
                'weights': quantized_weights
            }
        else:
            quantized_state_dict[k] = v
    return quantized_state_dict

In [ ]:
percentages = [0.1, 0.25,0.35, 0.45, 0.55, 0.65, 0.75, 0.85]  # Exemplo de porcentagens para teste
OPALA =[]
NISP = []
SNIP = []
OPALAmb=[]
NISPmb=[]
SNIPmb=[]

input_tensor = torch.rand(1, 3, 32, 32)  # Exemplo de tamanho de entrada para um modelo de visão computacional
global_model = SimpleModel(in_features=3, num_classes=10, dim=1600)
flops, params = profile(global_model, inputs=(input_tensor,))
print(f"FLOPs: {flops/10**6:.2f} MFLOPs")
print(f"Params: {params}")
def load_test_data(dataset, client_idx, batch_size=32):
    try:
        test_data = read_client_data(dataset, client_idx, is_train=False)
        X, y = zip(*test_data)
        X = torch.stack(X)
        y = torch.tensor(y)
        dataset = torch.utils.data.TensorDataset(X, y)
        return DataLoader(dataset, batch_size=batch_size)
    except Exception as e:
        print(f"Error loading test data: {e}")
        return None
for i in percentages:
    pruned_model = copy.deepcopy(global_model)
    g_model_pruned, mask = prune_and_restructure(model=pruned_model, pruning_rate=i, size_fc=25, data='Cifar10')
    size_before = sys.getsizeof(pickle.dumps(g_model_pruned))/ (1024 * 1024)
    pruned_flops, pruned_params = profile(g_model_pruned, inputs=(input_tensor,))
    print(f"Pruning Rate: {i*100:.0f}% - FLOPs: {pruned_flops/10**6:.2f} MFLOPs - Params: {pruned_params}")
    pruned_flops = pruned_flops / 10**6  # Convertendo para MFLOPs
    OPALA.append(pruned_flops)
    OPALAmb.append(size_before)

for i in percentages:
    a = []
    media = 0
    b = []
    for j in range(8):
        client_id = j
        pruned_model = copy.deepcopy(global_model)
        trainloader = load_test_data("Cifar10", client_id, 32)
                        
        g_model_pruned, _ = prune_fc1(model=pruned_model, 
                                            dataloader=trainloader, 
                                            pruning_ratio=i,
                                            device='cpu')
        g_model_pruned, mask = prune_and_restructure(model=g_model_pruned, pruning_rate=0.0, size_fc=25, data="Cifar10")
        size_before = sys.getsizeof(pickle.dumps(g_model_pruned))/ (1024 * 1024)
        b.append(size_before)
        pruned_flops, pruned_params = profile(g_model_pruned, inputs=(input_tensor,))
        pruned_flops = pruned_flops / 10**6 
        a.append(pruned_flops)
    soma = sum(b)
    media = soma / len(b)
    SNIPmb.append(media)
    soma = sum(a)
    media = soma / len(a)
    NISP.append(media)

for i in percentages:
    a = []
    b = []
    media = 0
    for j in range(8):
        client_id = j
        pruned_model = copy.deepcopy(global_model)
        trainloader = load_test_data("Cifar10", client_id, 32)
                            
        mask = snip_pruning(model=pruned_model, 
                                    dataloader=trainloader,
                                    criterion=nn.CrossEntropyLoss(), 
                                    pruning_ratio=i,
                                    device='cpu')
        g_model_pruned = apply_mask(pruned_model, mask)
        g_model_pruned, mask = prune_and_restructure(model=g_model_pruned, pruning_rate=0.0, size_fc=25, data="Cifar10")
        size_before = sys.getsizeof(pickle.dumps(g_model_pruned))/ (1024 * 1024)
        b.append(size_before)
        pruned_flops, pruned_params = profile(g_model_pruned, inputs=(input_tensor,))
        pruned_flops = pruned_flops / 10**6
        a.append(pruned_flops)
    soma = sum(b)
    media = soma / len(b)
    SNIPmb.append(media)
    soma = sum(a)
    media = soma / len(a)
    SNIP.append(media)

In [ ]:
print("OPALA:", OPALA)
print("OPALA MB:", OPALAmb)
print("NISP:", NISP)
print("NISP MB:", NISPmb)
print("SNIP:", SNIP)
print("SNIP MB:", SNIPmb)